# Exploratory Data Analysis — CFPB Consumer Complaints

This notebook explores the raw consumer complaint dataset before any processing — missing values, date ranges, category distributions, and basic cross-tabs that informed the preprocessing and feature engineering pipeline.

**Note:** This EDA was run on a 500,000-row sample of the raw CSV for speed. The production pipeline (`preprocessing.py` onward) processes the full dataset (15M+ rows), so exact counts here will differ from the dashboard.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

## 1. Load Data

In [ ]:
df = pd.read_csv(
    "../data/raw/complaints.csv",
    nrows=500000,
    low_memory=False
)

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.head(2)

In [ ]:
df.info()

## 2. Missing Values & Duplicates

In [ ]:
df.duplicated().sum()

In [ ]:
missing = df.isnull().sum()

missing_df = pd.DataFrame({
    "Missing_Count": missing,
    "Missing_Percentage": round((missing / len(df)) * 100, 2)
})

missing_df.sort_values("Missing_Percentage", ascending=False)

In [ ]:
df[df['Consumer complaint narrative'].isnull()].head(2)

## 3. Tags (Consumer Group)

In [ ]:
df["Tags"].value_counts(dropna=False)

In [ ]:
pd.crosstab(df["Tags"], df["Product"])

## 4. Date Features

Convert `Date received` to datetime, then derive Year/Month/Quarter/Day features used throughout the rest of the analysis and the dashboard.

In [ ]:
df["Date received"] = pd.to_datetime(
    df["Date received"],
    format="mixed",
    errors="coerce",
    utc=True
)

df["Date sent to company"] = pd.to_datetime(
    df["Date sent to company"],
    format="mixed",
    errors="coerce",
    utc=True
)

In [ ]:
df["Date received"].min(), df["Date received"].max()

In [ ]:
df["Date received"].isna().sum()

In [ ]:
df["Year"] = df["Date received"].dt.year
df["Month"] = df["Date received"].dt.month
df["Quarter"] = df["Date received"].dt.quarter
df["Day"] = df["Date received"].dt.day
df["Day_Name"] = df["Date received"].dt.day_name()

## 5. Complaint Volume Over Time

In [ ]:
df["Year"].value_counts().sort_index().plot(
    kind="bar",
    figsize=(10, 5),
    title="Complaints by Year",
    xlabel="Year",
    ylabel="Number of Complaints",
)
plt.tight_layout()
plt.show()

In [ ]:
df["Month"].value_counts().sort_index().plot(
    kind="bar",
    figsize=(10, 5),
    title="Complaints by Month",
    xlabel="Month",
    ylabel="Number of Complaints",
)
plt.tight_layout()
plt.show()

In [ ]:
df["Quarter"].value_counts().sort_index().plot(
    kind="bar",
    figsize=(8, 5),
    title="Complaints by Quarter",
    xlabel="Quarter",
    ylabel="Number of Complaints",
)
plt.tight_layout()
plt.show()

In [ ]:
df["Day_Name"].value_counts().plot(
    kind="bar",
    figsize=(10, 5),
    title="Complaints by Day of Week",
    xlabel="Day",
    ylabel="Number of Complaints",
)
plt.tight_layout()
plt.show()

In [ ]:
pd.crosstab(df["Year"], df["Product"])

In [ ]:
pd.crosstab(df["Year"], df["Company response to consumer"])

## 6. Product Analysis

In [ ]:
df["Product"].nunique()

In [ ]:
df["Product"].value_counts().head(10)

In [ ]:
pd.crosstab(df["Product"], df["Company response to consumer"])

In [ ]:
pd.crosstab(df["Product"], df["Timely response?"])

In [ ]:
product_pct = (
    df["Product"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

product_pct

## 7. Sub-Product

In [ ]:
df["Sub-product"].nunique()

In [ ]:
df["Sub-product"].value_counts().head(20)

In [ ]:
df["Sub-product"].isna().sum()

## 8. Timely Response

In [ ]:
df["Timely response?"].value_counts()

In [ ]:
(
    df["Timely response?"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

In [ ]:
pd.crosstab(df["Year"], df["Timely response?"])

## 9. Submission Channel

In [ ]:
df["Submitted via"].value_counts()

In [ ]:
(
    df["Submitted via"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

## 10. Company Analysis

In [ ]:
df["Company"].nunique()

In [ ]:
df["Company"].value_counts().head(20)

## 11. Geographic Analysis

**Note:** These are raw complaint counts by state, not normalized by state population — larger states will naturally show higher counts. Per-capita normalization is a planned improvement, not yet implemented here or in the dashboard.

In [ ]:
df["State"].nunique()

In [ ]:
df["State"].value_counts().head(20)

In [ ]:
state_product = pd.crosstab(df["State"], df["Product"])

top_product_by_state = state_product.idxmax(axis=1)
top_count_by_state = state_product.max(axis=1)

state_summary = pd.DataFrame({
    "Top_Product": top_product_by_state,
    "Complaint_Count": top_count_by_state
}).sort_values("Complaint_Count", ascending=False)

state_summary.head(20)

## 12. Issue Analysis

In [ ]:
df["Issue"].nunique()

In [ ]:
df["Issue"].value_counts(normalize=True).mul(100).round(2)

In [ ]:
issue_product = pd.crosstab(df["Issue"], df["Product"])

pd.DataFrame({
    "Top_Product": issue_product.idxmax(axis=1),
    "Count": issue_product.max(axis=1)
}).sort_values("Count", ascending=False)

In [ ]:
issue_company = pd.crosstab(df["Issue"], df["Company"])

pd.DataFrame({
    "Top_Company": issue_company.idxmax(axis=1),
    "Count": issue_company.max(axis=1)
}).sort_values("Count", ascending=False)

In [ ]:
top_issues = df["Issue"].value_counts().head(10).index

pd.crosstab(
    df[df["Issue"].isin(top_issues)]["Issue"],
    df["Product"]
)

In [ ]:
df["Sub-issue"].nunique()

In [ ]:
df["Sub-issue"].value_counts().head(20)

## 13. Resolution & Company Response

In [ ]:
df["Company response to consumer"].value_counts()

In [ ]:
df["Company public response"].value_counts()

In [ ]:
df["Resolution_Delay"] = (
    df["Date sent to company"] - df["Date received"]
).dt.days

df["Resolution_Delay"].describe()

In [ ]:
df["Resolution_Delay"].value_counts().head(10)

## 14. Narrative Text

Quick look at narrative availability and length. Deeper text analysis (top words, n-grams, topic modeling, classifiers) lives in `02_nlp_analysis.ipynb`.

In [ ]:
df["Consumer complaint narrative"].str.len().describe()

## 15. Sanity Check Against Processed Pipeline Output

Quick cross-check: does the yearly complaint distribution from this raw 500K-row sample roughly match the full processed dataset, and does the growth analysis output look sane? This is a spot-check, not a full validation.

In [ ]:
df1 = pd.read_parquet(
    "../data/processed/complaints_processed.parquet",
    columns=["Date received", "Year"]
)

df1["Year"].value_counts().sort_index()

In [ ]:
issue_growth = pd.read_parquet(
    "../data/processed/dashboard/issue_growth.parquet"
)

issue_growth[
    [
        "Issue",
        "Previous_Year_Complaints",
        "Current_Year_Complaints",
        "YoY_Growth_Pct",
        "Growth_Label"
    ]
].head(20)